## Future projections - Bezerra map

Use the projections from [Bezerra et al. 2022](https://doi.org/10.1371/journal.pone.0256052) to predict the future biomass accumulated by secondary forests.


In [1]:
import ee
import geemap
from utils import *

initialize()
config = ProjectConfig()
roi = config.roi
data_folder = config.data_folder
last_year = config.last_year

In [2]:
amazon = ee.FeatureCollection("projects/extents-490617/assets/biomes_br").filter(ee.Filter.eq('Bioma', 'Amazônia'))

ssp1_2015 = ee.Image("projects/forestregrowth/assets/projections/forest_2015_SSP1_RCP19")
ssp1_2050 = ee.Image("projects/forestregrowth/assets/projections/forest_2050_SSP1_RCP19")

ssp2_2015 = ee.Image("projects/forestregrowth/assets/projections/forest_2015_SSP2_RCP45")
ssp2_2050 = ee.Image("projects/forestregrowth/assets/projections/forest_2050_SSP2_RCP45")

ssp3_2015 = ee.Image("projects/forestregrowth/assets/projections/forest_2015_SSP3_RCP70")
ssp3_2050 = ee.Image("projects/forestregrowth/assets/projections/forest_2050_SSP3_RCP70")

# get the area increase per 10km grid cell
ssp1_delta = ssp1_2050.subtract(ssp1_2015).rename("ssp1_delta").clip(amazon.geometry())
ssp2_delta = ssp2_2050.subtract(ssp2_2015).rename("ssp2_delta").clip(amazon.geometry())
ssp3_delta = ssp3_2050.subtract(ssp3_2015).rename("ssp3_delta").clip(amazon.geometry())



Map(center=[0, 0], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI(childr…

In [5]:

image = ee.Image("projects/forestregrowth/assets/projections/forest_2015_SSP1_RCP19")

points = ee.FeatureCollection('projects/forestregrowth/assets/grid_1k_amazon_secondary')

projection = image.projection()

# 2. Convert points to an image by counting them per pixel/cell
#    reduceToImage assigns each point a value of 1, then reduceResolution sums them
points_with_value = points.map(lambda f: f.set('count', 1))

count_image = points_with_value.reduceToImage(
    properties=['count'],
    reducer=ee.Reducer.sum()
)

task = ee.batch.Export.image.toAsset(
    image = count_image,
    description = "count_image",
    assetId = "projects/forestregrowth/assets/count_image",
    region = amazon.geometry(),
    scale = image.projection().nominalScale(),
    maxPixels = 1e13,
    crs = projection
)
task.start()

In [ ]:

# make 10 predictions per 10km grid cell and average them out.
age = 35

# get average fire history per 10km grid cell

fire = (ee.Image("projects/mapbiomas-public/assets/brazil/fire/collection3/mapbiomas_fire_collection3_annual_burned_coverage_v1")
    .select([f"burned_coverage_{year}" for year in config.range_1985_2020])
    .byte()
    .rename([str(year) for year in config.range_1985_2020])
    .gt(0)
    .reduce('sum').rename("num_fires")).unmask(0)

sur_cover = ee.Image(f"{data_folder}/sur_cover") # should I get the 10k average as well?



mature_biomass_10k = ee.Image(f"{data_folder}/mature_biomass_10k")

# terraclim = ee.Image(f"{data_folder}/terraclim_1958_2019")



export_image = mature_biomass_10k.addBands(fire_10k).addBands(sur_cover_10k).addBands(ssp1_delta).addBands(ssp2_delta).addBands(ssp3_delta)

# map = geemap.Map()
# map.addLayer(export_image, {}, "Export Image")
# map.addLayer(mature_biomass_10k, {}, "Mature Biomass 10k")
# map.addLayer(ssp1_delta, {}, "SSP1 Delta")
# map




# extract using the 1km grid cell grid
# average predictions over 10km and get the area increase for that grid cell